In [ ]:
### ==========================================================================
### Two-Tower Retrieval Model for MovieLens 100K
### ==========================================================================
###
### This script builds a movie recommendation system using PyTorch.
###
### The idea: we learn a vector (embedding) for every user and every movie.
### If a user likes a movie, their vectors should point in the same direction
### (high dot product). If not, they should point away (low dot product).
###
### Architecture ("Two Towers"):
###   Tower 1 (User)  : user_id  -->  [Embedding table]  -->  32-dim vector
###   Tower 2 (Movie) : movie_id -->  [Embedding table] --+
###                     genres   -->  [19-dim binary]   --+--> Linear --> 32-dim vector
###                     avg_rating -> [1-dim scalar]    --+
###   Score = dot_product(user_vector, movie_vector)
###
### Loss: BPR (Bayesian Personalized Ranking)
###   For each (user, watched_movie) pair, we sample a random movie the user
###   has NOT watched. We then push the score of the watched movie higher
###   than the score of the unwatched movie.
###
### To recommend: compute user_vector dot ALL movie_vectors, pick the highest.
### ==========================================================================

# ── Install (only needed on Colab) ───────────────────────────────────────
# PyTorch and pandas come pre-installed on Colab, so nothing to install!

# ── Imports ──────────────────────────────────────────────────────────────

"""### 1. Imports and Hyperparameters"""

import os                                   # For file/folder operations
import zipfile                              # To unzip the downloaded dataset
import urllib.request                       # To download the dataset from the web
import random                               # For sampling random negative movies

import numpy as np                          # Numerical arrays
import pandas as pd                         # DataFrames for loading CSV data
import torch                                # PyTorch: the deep learning framework
import torch.nn as nn                       # Neural network building blocks
import torch.nn.functional as F             # Useful functions like logsigmoid
from torch.utils.data import Dataset, DataLoader  # For batching training data

# ── Hyperparameters ──────────────────────────────────────────────────────

SEED   = 42       # Random seed so results are reproducible every run
DIM    = 32       # Size of each embedding vector (higher = more expressive but slower)
BATCH  = 2048     # How many (user, movie) pairs per training step
EPOCHS = 10       # How many full passes through the training data
LR     = 1e-2     # Learning rate: how big each optimization step is (0.01)

# Set all random seeds so results are the same every time
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Use GPU if available (much faster), otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:

"""### 2. Load the Data"""

# URL where the MovieLens 100K dataset lives
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

# Download and unzip only if we haven't already
if not os.path.exists("ml-100k"):
    print("Downloading MovieLens 100K …")
    urllib.request.urlretrieve(url, "ml-100k.zip")       # Download the zip file
    with zipfile.ZipFile("ml-100k.zip") as z:
        z.extractall(".")                                # Unzip into current folder
    print("Done.")

# Load the ratings file: each row is (user_id, movie_id, rating, timestamp)
ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "movie", "rating", "ts"],
)

GENRE_COLS = [
    "unknown", "Action", "Adventure", "Animation", "Childrens", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "FilmNoir", "Horror",
    "Musical", "Mystery", "Romance", "SciFi", "Thriller", "War", "Western",
]
N_GENRES = len(GENRE_COLS)  # 19

# Load movie info: id, title, and 19 binary genre flags (columns 5-23)
titles = pd.read_csv(
    "ml-100k/u.item", sep="|", encoding="latin-1",
    header=None, usecols=[0, 1] + list(range(5, 24)),
    names=["movie", "title"] + GENRE_COLS,
)

# Merge so each rating row also has the movie's title
ratings = ratings.merge(titles, on="movie")


In [ ]:
"""### 3. Build Vocabularies"""

# Neural networks need numbers, not strings. So we map each unique user and
# movie title to a contiguous integer index (0, 1, 2, ...).

users  = sorted(ratings["user"].unique())    # List of all unique user IDs
movies = sorted(ratings["title"].unique())   # List of all unique movie titles

u2i = {u: i for i, u in enumerate(users)}   # user_id  -> integer index
m2i = {m: i for i, m in enumerate(movies)}  # title    -> integer index
i2m = {i: m for m, i in m2i.items()}         # integer  -> title (reverse lookup)

N_USERS  = len(users)    # Total number of unique users  (943)
N_MOVIES = len(movies)   # Total number of unique movies (1664)

# Build a genre tensor of shape (N_MOVIES, 19) aligned to movie indices.
# For each movie title, look up its genre row and store as a float tensor.
title_to_genre = titles.drop_duplicates(subset="title").set_index("title")[GENRE_COLS]
movie_genres = torch.zeros(N_MOVIES, N_GENRES)
for title, idx in m2i.items():
    if title in title_to_genre.index:
        movie_genres[idx] = torch.tensor(title_to_genre.loc[title].values, dtype=torch.float32)
movie_genres = movie_genres.to(device)

# Build a avg-rating tensor of shape (N_MOVIES, 1) aligned to movie indices.
# We compute the mean rating each movie received across all ratings, then
# normalise to [0, 1] by dividing by 5 (the max possible rating).
title_avg = ratings.groupby("title")["rating"].mean() / 5.0   # Series: title -> normalised avg
movie_avg_rating = torch.zeros(N_MOVIES, 1)
for title, idx in m2i.items():
    if title in title_avg.index:
        movie_avg_rating[idx, 0] = float(title_avg[title])
movie_avg_rating = movie_avg_rating.to(device)

# Build a set of movies each user has watched.
# We need this later to sample "negative" movies the user hasn't seen.
user_pos = {}                                           # dict: user_index -> set of movie_indices
for _, row in ratings.iterrows():                       # Loop through every rating
    uid = u2i[row["user"]]                              # Convert user ID to index
    mid = m2i[row["title"]]                             # Convert movie title to index
    user_pos.setdefault(uid, set()).add(mid)             # Add movie to this user's watched set

In [ ]:

"""### 4. Train / Test Split"""

# Randomly shuffle all 100k ratings, use the first 80k for training
# and the remaining 20k for testing.

perm     = np.random.permutation(len(ratings))                  # Random order
train_df = ratings.iloc[perm[:80_000]].reset_index(drop=True)   # First 80k = train
test_df  = ratings.iloc[perm[80_000:]].reset_index(drop=True)   # Last  20k = test

print(f"Users: {N_USERS}  Movies: {N_MOVIES}  Train: {len(train_df)}  Test: {len(test_df)}")


In [ ]:
"""### 5. Dataset with Negative Sampling"""

# For each real (user, watched_movie) pair, we also sample a random movie
# the user has NOT watched. This gives the model a "positive" and "negative"
# example to learn from.

class BPRDataset(Dataset):
    def __init__(self, df):
        self.pairs = [(u2i[r["user"]], m2i[r["title"]]) for _, r in df.iterrows()]
        self.genres     = movie_genres.cpu()         # Keep a CPU copy for DataLoader workers
        self.avg_rating = movie_avg_rating.cpu()     # Keep a CPU copy for DataLoader workers

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        u, pos = self.pairs[i]

        neg = random.randint(0, N_MOVIES - 1)
        while neg in user_pos[u]:
            neg = random.randint(0, N_MOVIES - 1)

        return (u,
                pos, self.genres[pos], self.avg_rating[pos],
                neg, self.genres[neg], self.avg_rating[neg])

# Create a DataLoader that shuffles and batches the training data
train_loader = DataLoader(
    BPRDataset(train_df),       # Our dataset
    batch_size=BATCH,           # 2048 samples per batch
    shuffle=True,               # Shuffle every epoch for better training
    drop_last=True,             # Drop incomplete last batch for consistent batch sizes
)

# Pre-compute test set as simple lists (no negative sampling needed for evaluation)
test_users  = [u2i[r["user"]]  for _, r in test_df.iterrows()]   # Test user indices
test_movies = [m2i[r["title"]] for _, r in test_df.iterrows()]   # Test movie indices (ground truth)

In [ ]:
"""### 6. The Two-Tower Model"""

# Two embedding tables: one for users, one for movies.
# Each table maps an integer index to a learned 32-dimensional vector.

class TwoTower(nn.Module):
    def __init__(self, n_users, n_movies, dim, n_genres=N_GENRES):
        super().__init__()
        self.user_emb   = nn.Embedding(n_users, dim)       # User  tower: n_users  x 32 table
        self.movie_emb  = nn.Embedding(n_movies, dim)       # Movie tower: n_movies x 32 table
        # Input: concat(movie_emb, genre_19d, avg_rating_1d)  ->  dim + n_genres + 1
        self.movie_proj = nn.Linear(dim + n_genres + 1, dim) # Projects back to 32-dim

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.movie_emb.weight)
        nn.init.xavier_uniform_(self.movie_proj.weight)

    def movie_vector(self, movie_idx, genre_feat, avg_rating):
        """Combine movie embedding with genre features and avg rating via a learned projection."""
        emb = self.movie_emb(movie_idx)                                        # (batch, dim)
        x   = torch.cat([emb, genre_feat, avg_rating], dim=-1)                 # (batch, dim+19+1)
        return self.movie_proj(x)                                               # (batch, dim)

# Create the model and move it to GPU (if available)
model = TwoTower(N_USERS, N_MOVIES, DIM, N_GENRES).to(device)

# Adam optimizer: adjusts model weights to minimize the loss
opt = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

In [ ]:
"""### 7. Evaluation Function"""

# After each epoch, we check: for each test (user, movie) pair, is the
# true movie in the model's top-K recommendations for that user?
# "top_100 = 0.40" means the true movie was in the top 100 picks 40% of the time.

@torch.no_grad()
def evaluate(ks=(1, 5, 10, 50, 100)):
    model.eval()

    # Pre-compute all movie vectors using embedding + genre + avg_rating projection
    all_idx = torch.arange(N_MOVIES, device=device)
    all_m   = model.movie_vector(all_idx, movie_genres, movie_avg_rating)  # (N_MOVIES, 32)

    hits  = {k: 0 for k in ks}
    total = len(test_users)

    t_users  = torch.tensor(test_users,  device=device)
    t_movies = torch.tensor(test_movies, device=device)

    for start in range(0, total, 4096):
        end = min(start + 4096, total)

        u_emb  = model.user_emb(t_users[start:end])
        scores = u_emb @ all_m.T
        true_m = t_movies[start:end]

        for k in ks:
            topk = scores.topk(k, dim=1).indices
            hits[k] += (topk == true_m.unsqueeze(1)).any(1).sum().item()

    return {k: hits[k] / total for k in ks}

In [ ]:
"""### 8. Training Loop"""

# Each epoch:
#   1. Loop through batches of (user, positive_movie, negative_movie)
#   2. Compute BPR loss: push score(user, positive) above score(user, negative)
#   3. Update weights via backpropagation
#   4. Evaluate top-K accuracy on the test set

# Print a header row for the results table
header = f"{'Ep':>3}  {'Loss':>8}  {'top1':>7}  {'top5':>7}  {'top10':>7}  {'top50':>7}  {'top100':>7}"
print(header)
print("-" * len(header))

for ep in range(1, EPOCHS + 1):
    model.train()                                      # Put model in training mode
    total_loss, steps = 0.0, 0                         # Track loss for this epoch

    for uids, pos, pos_g, pos_r, neg, neg_g, neg_r in train_loader:  # Each batch includes genre & avg_rating
        uids  = uids.to(device)
        pos   = pos.to(device)
        pos_g = pos_g.to(device)
        pos_r = pos_r.to(device)
        neg   = neg.to(device)
        neg_g = neg_g.to(device)
        neg_r = neg_r.to(device)

        u_emb = model.user_emb(uids)                        # Look up user embeddings:    (BATCH, 32)
        p_emb = model.movie_vector(pos, pos_g, pos_r)       # Positive movie vector:      (BATCH, 32)
        n_emb = model.movie_vector(neg, neg_g, neg_r)       # Negative movie vector:      (BATCH, 32)

        pos_score = (u_emb * p_emb).sum(1)                  # Dot product user·positive:  (BATCH,)
        neg_score = (u_emb * n_emb).sum(1)                  # Dot product user·negative:  (BATCH,)

        # BPR loss: we want pos_score > neg_score.
        # logsigmoid(pos - neg) is high when pos >> neg, low when pos ≈ neg.
        # We negate it because we want to MINIMIZE loss (maximize the gap).
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        opt.zero_grad()                                 # Clear old gradients
        loss.backward()                                 # Compute new gradients via backpropagation
        opt.step()                                      # Update model weights

        total_loss += loss.item()                       # Accumulate loss for logging
        steps += 1                                      # Count batches

    # Evaluate on test set after each epoch
    m = evaluate()
    print(f"{ep:>3}  {total_loss/steps:>8.4f}  "
          f"{m[1]:>7.4f}  {m[5]:>7.4f}  {m[10]:>7.4f}  {m[50]:>7.4f}  {m[100]:>7.4f}")

In [ ]:
"""### 9. Make Recommendations"""

# Given a user ID, compute their embedding, dot-product it with every movie
# embedding, and return the top-K highest-scoring movies.

@torch.no_grad()
def recommend(user_id, k=5):
    model.eval()

    u = model.user_emb(torch.tensor([u2i[user_id]], device=device))   # (1, 32)

    # Compute all movie vectors with genre features and avg ratings, then score
    all_idx = torch.arange(N_MOVIES, device=device)
    all_m   = model.movie_vector(all_idx, movie_genres, movie_avg_rating)  # (N_MOVIES, 32)
    scores  = (u @ all_m.T).squeeze(0)                                     # (N_MOVIES,)

    topk = scores.topk(k)
    return [(i2m[i.item()], s.item()) for i, s in zip(topk.indices, topk.values)]

# Show recommendations for two example users
print("\nTop 5 recommendations for user 42:")
for title, score in recommend(42):
    print(f"  score={score:+.3f}  {title}")

print("\nTop 5 recommendations for user 100:")
for title, score in recommend(100):
    print(f"  score={score:+.3f}  {title}")